# Centralized SegResNet32 training

Centralized SegResNet32 reference trained with ET-aware patch sampling and weighted Dice–cross-entropy loss. Checkpoint selection uses validation patches; full-volume test inference is handled in notebook 04.

In [1]:
from pathlib import Path
import os
import sys
from functools import partial

_start = Path(os.environ.get("BRATS_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / "src" / "brats_pipeline").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Place notebooks/ and src/ under one project root, or set BRATS_PROJECT_ROOT.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
from brats_pipeline.common import ProjectPaths
paths_io = ProjectPaths(PROJECT_ROOT)
resolve_existing_path = paths_io.resolve
print("Project:", PROJECT_ROOT)

Project: /home/mandrakedrink/projects/research_ts_bs


In [2]:
from brats_pipeline.common import seed_all, guard_output_directory
from brats_pipeline.data import (
    jp, load_nii as _load_nii, normalize_nonzero, remap_brats_labels,
    xyz_to_dhw, dhw_to_xyz, best_tumor_slice_dhw,
)
from brats_pipeline.inference import (
    create_segresnet, load_trusted_checkpoint, extract_model_state_dict,
    predict_labels_dhw, brats_region_dice_np, brats_region_voxels,
)
load_nii = partial(_load_nii, project_root=PROJECT_ROOT)
from brats_pipeline.training import (
    BratsManifestDatasetV2, WeightedDiceCELoss, dice_binary, brats_region_dice,
    validate_patch_level, get_round_lr, get_cpu_state_dict,
    fedavg_state_dicts, weighted_average_state_dicts,
    make_global_reference_state, fedprox_l2_penalty,
)

ALLOW_CHECKPOINT_OVERWRITE = False

In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nibabel as nib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [4]:
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram gb:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

torch: 2.12.0+cu126
cuda available: True
gpu: NVIDIA GeForce RTX 4090
vram gb: 23.5096435546875


In [5]:
seed_all(42, deterministic=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
try:
    import monai
    from monai.networks.nets import SegResNet
    from monai.losses import DiceLoss

    print("MONAI:", monai.__version__)
except ImportError:
    print("MONAI is not installed. Run: !pip install monai")

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


MONAI: 1.5.2


##Configuration

In [ ]:
CFG = {
    "manifest": str(PROJECT_ROOT / 'data/processed/manifest_4clients_seed42.csv'),

    # Input/training geometry.
    "patch_size": (128, 128, 128),
    # Batch size used on the 24 GB training GPU; accumulation gives 4 patches per optimizer step.
    "batch_size": 2,
    "grad_accum_steps": 2,
    "num_workers": 0,

    # Optimization.
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "epochs": 120,                  # max epochs, not fixed final length
    "min_epochs": 40,
    "early_stop_patience": 20,
    "early_stop_min_delta": 1e-4,
    "grad_clip_norm": 12.0,

    # Model.
    "num_classes": 4,
    "in_channels": 4,
    "init_filters": 32,

    # Patch-level validation during training.
    # Full-volume validation is done separately in notebook 04.
    "max_val_batches": None,

    # Mixed + ET-aware sampling probabilities.
    # Remapped labels: 3 == original BraTS label 4 / ET.
    "p_et": 0.20,
    "p_tumor": 0.50,
    "p_foreground": 0.20,
    "p_random": 0.10,

    # Lightweight 3D augmentations.
    "use_augmentation": True,
    "flip_prob": 0.50,
    "intensity_prob": 0.30,
    "scale_range": (0.90, 1.10),
    "shift_range": (-0.10, 0.10),
    "noise_prob": 0.20,
    "noise_std": 0.03,

    # Weighted CE inside Dice + CE loss.
    # Remapped labels:
    #   0 = background
    #   1 = original BraTS label 1 / tumor core
    #   2 = original BraTS label 2 / edema
    #   3 = original BraTS label 4 / enhancing tumor
    #
    # Background is down-weighted because it dominates voxel count.
    # ET is mildly up-weighted, but not too aggressively, because ET oversegmentation
    # was observed in small/absent ET cases.
    "use_ce_weight": True,
    "ce_weight": [0.10, 1.00, 1.00, 1.25],
    "dice_loss_weight": 1.0,
    "ce_loss_weight": 1.0,

    "save_dir": "models/centralized_segresnet32_final",
}

guard_output_directory(CFG["save_dir"], allow_overwrite=ALLOW_CHECKPOINT_OVERWRITE)

CFG

## Shared patch pipeline

Data loading, normalization, sampling and augmentations are imported from `brats_pipeline.data` and `brats_pipeline.training`.

## Manifest, splits, datasets, and loaders

In [ ]:
mf_df = pd.read_csv(CFG["manifest"])

print("manifest shape:", mf_df.shape)
mf_df.head()

In [ ]:
mf_df.groupby(["client_domain", "split"]).size()

In [ ]:
train_df = mf_df[mf_df["split"] == "train"].copy()
val_df = mf_df[mf_df["split"] == "val"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)

In [ ]:
train_ds = BratsManifestDatasetV2(
    train_df,
    patch_size=CFG['patch_size'],
    crop_mode='mixed',
    augment=CFG['use_augmentation'],
    config=CFG,
    project_root=PROJECT_ROOT
)
val_ds = BratsManifestDatasetV2(
    val_df,
    patch_size=CFG['patch_size'],
    crop_mode='tumor',
    augment=False,
    config=CFG,
    project_root=PROJECT_ROOT
)
print('train_ds:', len(train_ds))
print('val_ds:', len(val_ds))
print('train augmentation:', train_ds.augment)
print('val augmentation:', val_ds.augment)

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
batch = next(iter(train_loader))

print("image:", batch["image"].shape, batch["image"].dtype)
print("mask:", batch["mask"].shape, batch["mask"].dtype)
print("labels:", torch.unique(batch["mask"]))
print("crop_source:", batch["crop_source"])
print("patient:", batch["patient_id"])
print("domain:", batch["client_domain"])

## Model, weighted loss, optimizer, scheduler

In [ ]:
def create_model():
    return create_segresnet({
        "spatial_dims": 3, "in_channels": CFG["in_channels"],
        "out_channels": CFG["num_classes"], "init_filters": CFG["init_filters"],
        "blocks_down": (1, 2, 2, 4), "blocks_up": (1, 1, 1), "dropout_prob": 0.1,
    })

model = create_model().to(device)

sum_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("architecture:", f"SegResNet init_filters={CFG['init_filters']}")
print("total params:", f"{sum_params:,}")
print("trainable params:", f"{trainable_params:,}")

In [ ]:
if CFG["use_ce_weight"]:
    ce_weight = torch.tensor(
        CFG["ce_weight"],
        dtype=torch.float32,
        device=device,
    )
else:
    ce_weight = None

loss_fn = WeightedDiceCELoss(
    ce_weight=ce_weight,
    dice_weight=CFG["dice_loss_weight"],
    ce_loss_weight=CFG["ce_loss_weight"],
    include_background=True,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"],
    eta_min=1e-6,
)

use_amp = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

print("loss:", loss_fn.__class__.__name__)
print("CE weighting:", CFG["use_ce_weight"])
print("CE weights:", ce_weight)
print("dice_loss_weight:", CFG["dice_loss_weight"])
print("ce_loss_weight:", CFG["ce_loss_weight"])
print("optimizer:", optimizer.__class__.__name__)
print("scheduler:", scheduler.__class__.__name__)
print("AMP:", use_amp)
print("max epochs:", CFG["epochs"])
print("min epochs:", CFG["min_epochs"])
print("early stop patience:", CFG["early_stop_patience"])
print("batch size:", CFG["batch_size"])
print("grad accumulation:", CFG["grad_accum_steps"])
print("effective batch size:", CFG["batch_size"] * CFG["grad_accum_steps"])
print("grad clipping:", CFG["grad_clip_norm"])

In [9]:
def print_cuda_memory():
    """Print current and peak CUDA memory usage."""
    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3

    print(f"allocated:     {allocated:.2f} GB")
    print(f"reserved:      {reserved:.2f} GB")
    print(f"max allocated: {max_allocated:.2f} GB")


print_cuda_memory()

allocated:     0.00 GB
reserved:      0.00 GB
max allocated: 0.00 GB


## Forward diagnostic check

In [ ]:
model.train()

images = batch["image"].to(device, non_blocking=True)
masks = batch["mask"].to(device, non_blocking=True)

print("images:", images.shape, images.dtype)
print("masks:", masks.shape, masks.dtype)
print("mask labels:", torch.unique(masks))

with torch.amp.autocast("cuda", enabled=use_amp):
    logits = model(images)
    loss = loss_fn(logits, masks.unsqueeze(1))

print("logits:", logits.shape, logits.dtype)
print("loss:", float(loss.item()))

In [ ]:
@torch.no_grad()
def predict_batch(model, batch, device):
    """Predict integer labels for one batch without updating weights."""
    model.eval()

    images = batch["image"].to(device, non_blocking=True)

    with torch.amp.autocast("cuda", enabled=use_amp):
        logits = model(images)

    pred = torch.argmax(logits, dim=1)

    return pred.cpu()


def show_prediction_case(image_dhw, target_dhw, pred_dhw=None, title=""):
    """Display FLAIR, ground truth and an optional prediction."""
    z = best_tumor_slice_dhw(target_dhw)

    flair_slice = image_dhw[3, z]
    target_slice = target_dhw[z]

    if pred_dhw is None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    else:
        pred_slice = pred_dhw[z]
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(flair_slice, cmap="gray")
    axes[0].set_title("FLAIR")
    axes[0].axis("off")

    axes[1].imshow(flair_slice, cmap="gray")
    axes[1].imshow(target_slice, alpha=0.35, vmin=0, vmax=3)
    axes[1].set_title("Ground truth")
    axes[1].axis("off")

    if pred_dhw is not None:
        axes[2].imshow(flair_slice, cmap="gray")
        axes[2].imshow(pred_slice, alpha=0.35, vmin=0, vmax=3)
        axes[2].set_title("Prediction")
        axes[2].axis("off")

    fig.suptitle(f"{title} | slice={z}")
    plt.tight_layout()
    plt.show()

## Single-batch optimization check

This diagnostic changes model weights and consumes random numbers. The model and optimizer are reset below before the main run.

In [ ]:
def overfit_one_batch(model, batch, loss_fn, optimizer, scaler, device, steps=30):
    """Run a short single-batch optimization check."""
    model.train()

    images = batch["image"].to(device, non_blocking=True)
    masks = batch["mask"].to(device, non_blocking=True)

    losses = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks.unsqueeze(1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.item()))

        if step % 5 == 0 or step == steps - 1:
            print(f"step {step:03d} | loss={loss.item():.4f}")

    return losses

In [ ]:
losses = overfit_one_batch(
    model=model,
    batch=batch,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scaler=scaler,
    device=device,
    steps=30,
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses, marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Overfit one batch sanity check")
plt.grid(alpha=0.3)
plt.show()

## Training and validation functions

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, scaler, device):
    """Train one epoch with AMP, gradient accumulation and clipping."""
    model.train()

    losses = []
    crop_sources = []

    grad_accum_steps = CFG.get("grad_accum_steps", 1)
    grad_clip_norm = CFG.get("grad_clip_norm", None)

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(images)
            loss = loss_fn(logits, masks.unsqueeze(1))
            loss_for_backward = loss / grad_accum_steps

        scaler.scale(loss_for_backward).backward()

        should_step = (
            ((step + 1) % grad_accum_steps == 0) or
            ((step + 1) == len(loader))
        )

        if should_step:
            if grad_clip_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=grad_clip_norm,
                )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(float(loss.item()))

        if "crop_source" in batch:
            crop_sources.extend(list(batch["crop_source"]))

    mean_loss = float(np.mean(losses))

    if len(crop_sources) > 0:
        crop_source_counts = pd.Series(crop_sources).value_counts().to_dict()
    else:
        crop_source_counts = {}

    return mean_loss, crop_source_counts

## Reset model and optimizer

In [ ]:
model = create_model().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"],
    eta_min=1e-6,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

total_params = sum(p.numel() for p in model.parameters())
print("model reset")
print("architecture:", f"SegResNet init_filters={CFG['init_filters']}")
print("total params:", f"{total_params:,}")

## Centralized final training

In [ ]:
history = []
best_val_loss = float('inf')
best_val_mean_dice = -float('inf')
best_epoch_mean_dice = None
epochs_without_improvement = 0
for epoch in range(1, CFG['epochs'] + 1):
    train_loss, crop_source_counts = train_one_epoch(
        model=model,
        loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        scaler=scaler,
        device=device
    )
    val_loss, val_dice = validate_patch_level(
        model=model,
        loader=val_loader,
        loss_fn=loss_fn,
        device=device,
        max_batches=CFG['max_val_batches'],
        use_amp=use_amp
    )
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    val_mean_dice = (val_dice['WT'] + val_dice['TC'] + val_dice['ET']) / 3.0
    row = {
        'epoch': epoch,
        'lr': current_lr,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_dice_WT': val_dice['WT'],
        'val_dice_TC': val_dice['TC'],
        'val_dice_ET': val_dice['ET'],
        'val_mean_dice': val_mean_dice,
        'crop_source_counts': json.dumps(crop_source_counts)
    }
    history.append(row)
    improved_mean_dice = val_mean_dice > best_val_mean_dice + CFG['early_stop_min_delta']
    print(f"epoch {epoch:03d} | lr={current_lr:.2e} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | WT={val_dice['WT']:.4f} | TC={val_dice['TC']:.4f} | ET={val_dice['ET']:.4f} | mean={val_mean_dice:.4f} | no_improve={epochs_without_improvement} | crops={crop_source_counts}")
    if improved_mean_dice:
        best_val_mean_dice = val_mean_dice
        best_epoch_mean_dice = epoch
        epochs_without_improvement = 0
        ckpt_path = os.path.join(CFG['save_dir'], 'segresnet_final_best_mean_dice.pt')
        torch.save(
            {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'epoch': epoch,
                'cfg': CFG,
                'best_val_mean_dice': best_val_mean_dice,
                'history': history
            },
            ckpt_path
        )
        print('saved best mean dice:', ckpt_path)
    else:
        epochs_without_improvement += 1
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = os.path.join(CFG['save_dir'], 'segresnet_final_best_loss.pt')
        torch.save(
            {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'epoch': epoch,
                'cfg': CFG,
                'best_val_loss': best_val_loss,
                'history': history
            },
            ckpt_path
        )
        print('saved best loss:', ckpt_path)
    last_path = os.path.join(CFG['save_dir'], 'segresnet_final_last.pt')
    torch.save(
        {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch,
            'cfg': CFG,
            'history': history,
            'best_epoch_mean_dice': best_epoch_mean_dice,
            'best_val_mean_dice': best_val_mean_dice
        },
        last_path
    )
    if epoch >= CFG['min_epochs'] and epochs_without_improvement >= CFG['early_stop_patience']:
        print(f'Early stopping at epoch {epoch}. Best epoch: {best_epoch_mean_dice}, best val_mean_dice={best_val_mean_dice:.4f}')
        break

## Save history and diagnostic plots

In [ ]:
hist = pd.DataFrame(history)

hist_path = os.path.join(
    CFG["save_dir"],
    "segresnet_final_history.csv",
)

hist.to_csv(hist_path, index=False)

cfg_path = os.path.join(
    CFG["save_dir"],
    "segresnet_final_config.json",
)

with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)

print("saved history:", hist_path)
print("saved config:", cfg_path)

hist.tail()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
plt.plot(hist["epoch"], hist["val_loss"], marker="o", label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Centralized SegResNet final training loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(hist["epoch"], hist["val_dice_WT"], marker="o", label="WT")
plt.plot(hist["epoch"], hist["val_dice_TC"], marker="o", label="TC")
plt.plot(hist["epoch"], hist["val_dice_ET"], marker="o", label="ET")
plt.plot(hist["epoch"], hist["val_mean_dice"], marker="o", linestyle="--", label="Mean Dice")
plt.xlabel("Epoch")
plt.ylabel("Patch-level Dice")
plt.title("Validation Dice by BraTS region — centralized final")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Best checkpoint summary

In [ ]:
best_by_mean = hist.loc[hist["val_mean_dice"].idxmax()]
best_by_loss = hist.loc[hist["val_loss"].idxmin()]

print("Best by mean Dice:")
print(best_by_mean)

print("\nBest by val loss:")
print(best_by_loss)

## Prediction diagnostic after training

In [ ]:
batch = next(iter(val_loader))
pred = predict_batch(model, batch, device)

print("pred shape:", pred.shape)
print("pred labels:", torch.unique(pred))
print("target labels:", torch.unique(batch["mask"]))

In [ ]:
img0 = batch["image"][0].cpu().numpy()
gt0 = batch["mask"][0].cpu().numpy()
pred0 = pred[0].numpy()

show_prediction_case(
    image_dhw=img0,
    target_dhw=gt0,
    pred_dhw=pred0,
    title=batch["patient_id"][0],
)